# Evaluation — Myanmar ASR & English/Korean Translation

Adapted from the evaluation methods in the two supplied Gemma 4 zero-shot notebooks. Upload reference and prediction JSONL files, then calculate **CER, BLEU, chrF++**. This notebook evaluates saved predictions; it does not run Gemma.

**Note:** Only compare models evaluated on the same language, subset, and settings.

In [ ]:
%pip -q install jiwer sacrebleu pandas

## 1. Upload reference and prediction files
Run the cell and upload **two JSONL files**: the reference metadata and model predictions. For Korean, use the Korean reference metadata and Korean predictions.

# Evaluation Metric Calculations

This section explains the exact metrics used by this notebook.

## 1. ASR — Character Error Rate (CER)

$$CER = \frac{S + D + I}{N}$$

- **S**: substitutions
- **D**: deletions
- **I**: insertions
- **N**: number of characters in the reference Myanmar transcription
- **Lower is better.** CER = 0 means an exact character-level match after normalization.

## 2. Translation — BLEU

BLEU compares the predicted English/Korean translation with the reference translation using n-gram overlap. This notebook uses SacreBLEU. English uses the `13a` tokenizer with `effective_order=True`; Korean uses `tokenize="none"`.

- **Higher is better.**

## 3. Translation — chrF++

chrF++ compares the predicted translation with the reference using character n-grams and word n-grams. This notebook uses SacreBLEU `CHRF(word_order=2)`.

- **Higher is better.**


In [ ]:
# ============================================================
# METRIC CALCULATION FUNCTIONS
# ============================================================

from jiwer import cer
from sacrebleu.metrics import BLEU, CHRF

def calculate_cer(reference_texts, predicted_texts):
    """CER = (Substitutions + Deletions + Insertions) / reference characters."""
    return cer(reference_texts, predicted_texts)

def calculate_bleu(reference_texts, predicted_texts, language):
    """Corpus BLEU using the same settings as the official evaluation below."""
    metric = BLEU(tokenize="13a", effective_order=True) if language == "en" else BLEU(tokenize="none")
    return metric.corpus_score(predicted_texts, [reference_texts]).score

def calculate_chrfpp(reference_texts, predicted_texts):
    """chrF++ using character n-grams plus word n-grams (word_order=2)."""
    metric = CHRF(word_order=2)
    return metric.corpus_score(predicted_texts, [reference_texts]).score

print("Metric functions ready: CER, BLEU, chrF++")


In [ ]:
from google.colab import files
from pathlib import Path
print("Upload your reference JSONL and predictions JSONL files")
uploaded = files.upload()
print("Uploaded:", list(uploaded))

## 2. Select language, files, and evaluation subset
Change the three settings below to match your experiment. **Korean primary_approved** is different from the full evaluation set.

In [ ]:
LANGUAGE = "en"  # "en" or "ko"
REFERENCE_FILE = "test_myanmar_english_translated.jsonl"  # replace with uploaded filename
PREDICTION_FILE = "predictions.jsonl"  # replace with uploaded filename
SUBSET = "all"  # "all", "primary_approved", "approved_only", "provisional_secondary"

## 3. Calculate evaluation scores

In [ ]:
"""Evaluate Gemma 4 Myanmar ASR and English/Korean speech translation.

Adapted from the evaluation cells of the supplied English and Korean notebooks.
Install: pip install jiwer sacrebleu
Example:
 python evaluate.py --language en --references test_myanmar_english_translated.jsonl --predictions predictions.jsonl
 python evaluate.py --language ko --references test_myanmar_korean_translated.full.jsonl --predictions predictions.jsonl --subset primary_approved
"""
import json
import unicodedata
from pathlib import Path
from jiwer import cer
from sacrebleu.metrics import BLEU, CHRF


def read_jsonl(path):
    with Path(path).open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def normalize_my(text, language):
    # Preserve the distinct Myanmar normalization used in each source notebook.
    form = 'NFKC' if language == 'en' else 'NFC'
    text = unicodedata.normalize(form, str(text or ''))
    return ''.join(c for c in text if not c.isspace() and not unicodedata.category(c).startswith('P') and (language != 'en' or not unicodedata.category(c).startswith('Z')))


def normalize_ko(text):
    return ' '.join(unicodedata.normalize('NFC', str(text or '')).split())




def evaluate(language, reference_file, prediction_file, subset="all"):
    references = list(read_jsonl(reference_file))
    predictions = {r['sample_id']: r for r in read_jsonl(prediction_file)}
    if len({r['sample_id'] for r in references}) != len(references):
        raise ValueError("Duplicate sample_id in references")
    if subset != "all":
        status = "approved_reference" if subset in ("approved_only", "primary_approved") else "provisional_reference"
        key = "translation_status" if language == "en" else "translation_ko_status"
        references = [r for r in references if r.get(key) == status]
    translation_key = "translation_en" if language == "en" else "translation_ko"
    prediction_key = "gemma_s2tt_en" if language == "en" else "gemma_s2tt_ko"
    bleu = BLEU(tokenize="13a", effective_order=True) if language == "en" else BLEU(tokenize="none")
    chrf = CHRF(word_order=2)
    translation_refs, translation_hyps, asr_refs, asr_hyps = [], [], [], []
    successful = 0
    for ref in references:
        pred = predictions.get(ref["sample_id"])
        if not pred or pred.get("generation_status") != "success":
            continue
        successful += 1
        translation = pred.get(prediction_key)
        if isinstance(translation, str) and ref.get(translation_key):
            r, h = str(ref[translation_key]), translation
            if language == "ko":
                r, h = normalize_ko(r), normalize_ko(h)
            translation_refs.append(r)
            translation_hyps.append(h)
        asr = pred.get("gemma_asr_my")
        if isinstance(asr, str) and normalize_my(ref.get("transcription_my"), language):
            if language == "ko" or asr.strip():
                asr_refs.append(normalize_my(ref["transcription_my"], language))
                asr_hyps.append(normalize_my(asr, language))
    return {
        "language": language, "subset": subset, "expected": len(references),
        "successful_generations": successful, "translation_scored": len(translation_refs),
        "asr_scored": len(asr_refs),
        "asr_cer": cer(asr_refs, asr_hyps) if asr_refs else None,
        "bleu": bleu.corpus_score(translation_hyps, [translation_refs]).score if translation_refs else None,
        "chrf++": chrf.corpus_score(translation_hyps, [translation_refs]).score if translation_refs else None,
        "bleu_tokenizer": "13a" if language == "en" else "none",
        "chrf_word_order": 2
    }

results = evaluate(LANGUAGE, REFERENCE_FILE, PREDICTION_FILE, SUBSET)
print(json.dumps(results, ensure_ascii=False, indent=2))

## 4. Download evaluation results

In [ ]:
from google.colab import files
Path("evaluation_results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
files.download("evaluation_results.json")